# Blending

Workbench for combining the individual experiment runs in `experiments/runs.csv` into
stronger predictions. Planned uses:

- **Ensembling** — simple/weighted averages of model probabilities (and rank averages).
- **Hill climbing** — greedily grow a weighted blend on the OOF predictions, adding the
  model that most improves the OOF metric at each step (Caruana-style ensemble selection).
- **Stacking** — train a meta-learner on the out-of-fold (OOF) prediction columns.

All of these need the per-run **out-of-fold predictions** (`oof_proba.npy`), which give a
leakage-free prediction for every training row under the shared
`StratifiedKFold(5, shuffle=True, random_state=42)` split.

This first cut just measures **how similar the runs are to each other**, pairwise, via the
Pearson and Spearman correlation of their OOF probabilities. Blending pays off most when the
members are individually strong *and* mutually **de-correlated**, so this is the natural
first diagnostic.

## What is Spearman's correlation coefficient?

**Pearson's** $r$ measures the strength of a *linear* relationship between two variables — it
works on the raw values and asks "do they move up and down together, proportionally?"

**Spearman's** $\rho$ (rho) is just **Pearson's $r$ computed on the ranks** of the values
instead of the values themselves. You replace each column by `1, 2, 3, …` according to sort
order, then correlate those ranks. As a result it measures whether the two variables move
together **monotonically** — when one goes up, does the other tend to go up — *regardless of
whether the relationship is a straight line*.

$\rho = 1$ means the two runs rank every customer in the **exact same order**; $\rho = -1$
means perfectly reversed; $0$ means no monotonic relationship.

Why it matters here:

- The competition metric (ROC-AUC) and rank-averaging both depend **only on the ordering** of
  the predicted probabilities, not their absolute calibration. Two models can disagree on the
  raw probabilities (low Pearson) yet rank customers near-identically (high Spearman) — and
  for an AUC blend, it is the **Spearman** agreement that tells you how much fresh signal a
  second model actually adds.
- Spearman is **invariant to any monotonic rescaling** (e.g. one model being systematically
  over-confident), so it is a cleaner diversity measure for ranking-based ensembles, while
  Pearson is the relevant one for plain probability averaging.

In [ ]:
import os

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

# The canonical training-set size. Runs whose OOF array is shorter were trained on a
# subsample (e.g. the TabPFN/TabICL 50k EDA passes) and their rows do NOT align with the
# full-set runs, so they cannot be correlated row-wise and are excluded below.
CANON_N = 594_194

runs = pd.read_csv("experiments/runs.csv").dropna(subset=["run_id"])
runs = runs[runs["status"] == "success"].reset_index(drop=True)

oof_cols, labels, kept = [], [], []
skipped = []
seen_tags = set()
for _, r in runs.iterrows():
    path = os.path.join(str(r["artifact_dir"]).replace("\\", "/"), "oof_proba.npy")
    proba = np.load(path)
    if proba.shape[0] != CANON_N:
        skipped.append((r["tag"], proba.shape[0]))
        continue
    # Disambiguate runs that share a tag (e.g. lr-pipeline on fe_v0 vs fe_v1).
    label = r["tag"] if r["tag"] not in seen_tags else f"{r['tag']}|{r['data_version']}"
    seen_tags.add(r["tag"])
    oof_cols.append(proba)
    labels.append(label)
    kept.append(r)

oof = pd.DataFrame(np.column_stack(oof_cols), columns=labels)
meta = pd.DataFrame(kept)[["tag", "model_class", "data_version", "oof_roc_auc", "run_id"]]
meta.insert(0, "label", labels)

print(f"OOF matrix: {oof.shape[0]:,} rows x {oof.shape[1]} runs")
if skipped:
    print("Excluded (subsample OOF, rows do not align):")
    for tag, n in skipped:
        print(f"  - {tag}: {n:,} rows")

In [ ]:
# Pairwise correlation matrices across the run OOF probabilities.
# Pearson: linear agreement on the raw probabilities (relevant to probability averaging).
# Spearman: rank agreement (relevant to AUC / rank-averaging diversity).
pearson = oof.corr(method="pearson")

rho, _ = spearmanr(oof.values)          # one rank-and-correlate pass over all columns
spearman = pd.DataFrame(rho, index=labels, columns=labels)

def show(corr, title):
    print(title)
    styled = (corr.style
              .background_gradient(cmap="RdYlGn_r", vmin=corr.values.min(), vmax=1.0)
              .format("{:.4f}"))
    return styled

show(spearman, "Spearman (rank) correlation of OOF probabilities")

In [ ]:
show(pearson, "Pearson (linear) correlation of OOF probabilities")

### Most de-correlated pairs

The pairs with the **lowest** Spearman correlation are the most promising blend partners —
they rank customers most differently, so each carries signal the other misses. (Pair them
with the leaderboard OOF ROC-AUC to balance diversity against individual strength.)

In [ ]:
def ranked_pairs(corr, ascending=True, n=15):
    m = corr.values.copy()
    iu = np.triu_indices_from(m, k=1)
    pairs = pd.DataFrame({
        "run_a": [corr.index[i] for i in iu[0]],
        "run_b": [corr.columns[j] for j in iu[1]],
        "spearman": m[iu],
    })
    pairs = pairs.sort_values("spearman", ascending=ascending).reset_index(drop=True)
    return pairs.head(n)

least = ranked_pairs(spearman, ascending=True, n=15)
least["spearman"] = least["spearman"].round(4)
least.style.hide(axis="index")